# 1.4 · MCP vs. Skills vs. Functions: who does what

**MCP、技能（Skills）与函数调用的区别**

> **After this lesson you can:** Tell function calling, MCP and Agent Skills apart in one sentence each, pick the right one for a requirement, and show how the three stack in one agent.

Official reading: [Agent Skills specification](https://agentskills.io/specification) · [Skills extension](https://modelcontextprotocol.io/extensions/skills/overview) · [Build with Agent Skills](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-with-agent-skills) · [Build an MCP client (function calling loop)](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-client)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. One task, three layers
We'll build the same small job three ways — *“check an expense against company policy”* — so the differences are concrete. First, the **function calling** version: a Python function plus the tool definition you would send in a Claude `tools=[...]` list. Everything lives in this notebook, which stands in for “your app”.

In [ ]:
import json

POLICY = {"meals": 60, "hotel": 220, "taxi": 80}          # max EUR per item


def check_expense(category: str, amount: float) -> dict:
    """The business logic: is one expense within policy?"""
    limit = POLICY.get(category)
    if limit is None:
        return {"ok": False, "reason": f"unknown category {category!r}"}
    return {"ok": amount <= limit, "limit": limit, "over_by": max(0, amount - limit)}


# Function calling: YOU write the definition the model sees, by hand, inside your app.
FUNCTION_TOOL = {
    "name": "check_expense",
    "description": "Check one expense against company policy. Returns ok, the limit and how much it is over.",
    "input_schema": {
        "type": "object",
        "properties": {"category": {"type": "string", "enum": sorted(POLICY)},
                       "amount": {"type": "number", "description": "Amount in EUR"}},
        "required": ["category", "amount"],
    },
}

# What the model sends back when it wants the action (shape of a Claude `tool_use` block):
tool_use = {"type": "tool_use", "id": "toolu_01", "name": "check_expense",
            "input": {"category": "hotel", "amount": 260}}

# Your code executes it locally and returns a `tool_result` block:
result = check_expense(**tool_use["input"])
print(json.dumps({"type": "tool_result", "tool_use_id": tool_use["id"], "content": json.dumps(result)}, indent=2))

### 2. The same capability as an MCP server
Now the logic moves into an **MCP server**. The host no longer hard-codes the definition: it *discovers* it with `tools/list` and converts it into the model's tool format. Notice the converted definition has exactly the same shape as `FUNCTION_TOOL` — **MCP feeds function calling, it doesn't replace it.**

In [ ]:
from typing import Annotated, Literal

from pydantic import Field

from mcp import Client
from mcp.server import MCPServer

finance = MCPServer("finance")


@finance.tool()
def check_expense_tool(
    category: Annotated[Literal["hotel", "meals", "taxi"], Field(description="Expense category")],
    amount: Annotated[float, Field(description="Amount in EUR")],
) -> dict[str, object]:
    """Check one expense against company policy. Returns ok, the limit and how much it is over."""
    return check_expense(category, amount)


@finance.tool()
def list_expenses(employee: str) -> list[dict[str, object]]:
    """List last week's expenses for an employee."""
    return [{"id": 1, "category": "hotel", "amount": 300.0},
            {"id": 2, "category": "meals", "amount": 35.5},
            {"id": 3, "category": "taxi", "amount": 95.0}]


async with Client(finance) as client:
    mcp_tools = (await client.list_tools()).tools

# The host's one-line adapter: MCP tool -> model tool definition
as_model_tools = [{"name": t.name, "description": t.description, "input_schema": t.input_schema} for t in mcp_tools]
print(json.dumps(as_model_tools[0], indent=2))
print("\nsame keys as the hand-written function tool:", as_model_tools[0].keys() == FUNCTION_TOOL.keys())

### 3. The know-how as an Agent Skill
The tools say *what can be done*. They don't say *how the company wants an expense review done*: check every item, round to cents, cite the policy, escalate anything more than 25 % over. That procedure goes into a **skill**: a folder with `SKILL.md` and a reference file, written to a temp directory.

In [ ]:
import tempfile
import textwrap
from pathlib import Path

skills_root = Path(tempfile.mkdtemp()) / "skills"
skill_dir = skills_root / "expense-review"
(skill_dir / "references").mkdir(parents=True)

(skill_dir / "SKILL.md").write_text(textwrap.dedent("""\
    ---
    name: expense-review
    description: Review an employee's expense report against company policy and flag violations. Use when asked to audit, check or approve expenses.
    ---
    # Expense review

    1. Call `list_expenses` for the employee.
    2. For EVERY item call `check_expense_tool` (never guess limits).
    3. If an item is more than 25% over its limit, mark it ESCALATE; otherwise mark it FLAG.
       The escalation rules are in references/escalation.md - read it only if something must be escalated.
    4. Finish with a table: id, category, amount, verdict.
"""))
(skill_dir / "references" / "escalation.md").write_text(
    "Escalations go to finance-approvals@example.com with the receipt attached, within 2 working days.\n")

for path in sorted(skill_dir.rglob("*")):
    print(path.relative_to(skills_root))

### 4. A tiny skill loader with progressive disclosure
A host that supports skills does three things: validate the frontmatter, put **only metadata** in the model's context, and load the body or a reference file **only on demand**. Watch the token estimate (≈ characters ÷ 4) at each level.

In [ ]:
import re

NAME_RULE = re.compile(r"^[a-z0-9]([a-z0-9-]{0,62}[a-z0-9])?$")   # 1-64 chars, a-z 0-9 -, no edge hyphens


def parse_skill(skill_md: Path) -> dict:
    text = skill_md.read_text()
    match = re.match(r"---\n(.*?)\n---\n(.*)", text, re.S)
    meta = dict(line.split(": ", 1) for line in match.group(1).splitlines())
    assert NAME_RULE.match(meta["name"]), f"invalid skill name {meta['name']!r}"
    assert 1 <= len(meta["description"]) <= 1024, "description must be 1-1024 characters"
    assert meta["name"] == skill_md.parent.name, "name should match the directory name"
    return {"name": meta["name"], "description": meta["description"], "body": match.group(2), "dir": skill_md.parent}


tokens = lambda s: len(s) // 4
catalog = {s["name"]: s for s in (parse_skill(p) for p in skills_root.glob("*/SKILL.md"))}

level1 = "\n".join(f"- {s['name']}: {s['description']}" for s in catalog.values())
print("LEVEL 1 - always in context (metadata):", tokens(level1), "tokens\n" + level1)
print("\nLEVEL 2 - on activation (SKILL.md body):", tokens(catalog["expense-review"]["body"]), "tokens")
ref = (catalog["expense-review"]["dir"] / "references" / "escalation.md").read_text()
print("LEVEL 3 - only if a step needs it (references/escalation.md):", tokens(ref), "tokens")
print("\nfor comparison, the MCP tool definitions sent with EVERY request:", tokens(json.dumps(as_model_tools)), "tokens")

### 5. All three in one agent run
Now an agent loop that uses everything. A scripted stand-in for the model (so it runs offline) reads the skill catalog, activates `expense-review`, follows its steps by emitting `tool_use` calls, and the host executes each one through MCP. The trace labels which layer did each step.

In [ ]:
def scripted_model(state: dict) -> dict:
    """A deterministic stand-in for the LLM that follows the skill's procedure step by step."""
    if "skill" not in state:
        return {"action": "activate_skill", "name": "expense-review"}            # chosen from level-1 metadata
    if "expenses" not in state:
        return {"action": "tool_use", "name": "list_expenses", "input": {"employee": "ana"}}
    pending = [e for e in state["expenses"] if e["id"] not in state["checked"]]
    if pending:
        e = pending[0]
        return {"action": "tool_use", "name": "check_expense_tool", "input": {"category": e["category"], "amount": e["amount"]}, "id": e["id"]}
    if any(v["over_by"] > 0.25 * v["limit"] for v in state["checked"].values()) and "escalation" not in state:
        return {"action": "read_skill_file", "path": "references/escalation.md"}
    return {"action": "final"}


state = {"checked": {}}
async with Client(finance) as client:
    while (step := scripted_model(state))["action"] != "final":
        if step["action"] == "activate_skill":
            state["skill"] = catalog[step["name"]]["body"]
            print(f"[SKILLS level 2]  load SKILL.md of {step['name']!r}")
        elif step["action"] == "read_skill_file":
            state["escalation"] = (catalog["expense-review"]["dir"] / step["path"]).read_text()
            print(f"[SKILLS level 3]  read {step['path']}")
        else:
            print(f"[FUNCTION CALL]   model emits tool_use {step['name']}({step['input']})")
            result = await client.call_tool(step["name"], step["input"])
            print(f"[MCP]             host sends tools/call to server 'finance' -> {result.structured_content}")
            if step["name"] == "list_expenses":
                state["expenses"] = result.structured_content["result"]   # lists are wrapped in {"result": ...}
            else:
                state["checked"][step["id"]] = result.structured_content     # a dict return is the object itself

print("\nid | category | amount | verdict")
for e in state["expenses"]:
    v = state["checked"][e["id"]]
    verdict = "OK" if v["ok"] else ("ESCALATE" if v["over_by"] > 0.25 * v["limit"] else "FLAG")
    print(f"{e['id']}  | {e['category']:<8} | {e['amount']:>6.2f} | {verdict}")

Read the trace: the **skill** decided the procedure (check every item, read the escalation file only when needed), **function calling** expressed each action, and **MCP** carried each action to the server that owns the logic.

### 6. Optional: the real model chooses
With `ANTHROPIC_API_KEY` set, Claude gets the skill catalog in the system prompt, a `load_skill` function for level 2, and the finance MCP tools — then decides on its own when to load the skill and which tools to call. Without a key the cell explains what would happen.

In [ ]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    load_skill = {"name": "load_skill", "description": "Load the full instructions of a skill by name.",
                  "input_schema": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}}
    system = "You can use these skills; call load_skill before following one.\n" + level1
    messages = [{"role": "user", "content": "Please audit ana's expenses from last week."}]
    async with Client(finance) as client:
        for _ in range(12):
            response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, system=system,
                                           tools=as_model_tools + [load_skill], messages=messages)
            if response.stop_reason != "tool_use":
                break
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                if block.name == "load_skill":
                    print("[SKILLS] Claude loaded", block.input)
                    content, is_error = catalog.get(block.input["name"], {}).get("body", "no such skill"), False
                else:
                    print("[MCP]    Claude called", block.name, block.input)
                    r = await client.call_tool(block.name, block.input)
                    content, is_error = json.dumps(r.structured_content), r.is_error
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": content, "is_error": is_error})
            messages.append({"role": "user", "content": results})
    print("\n".join(b.text for b in response.content if b.type == "text"))
else:
    print("skipped: set ANTHROPIC_API_KEY to watch Claude pick the skill from its description, "
          "load it, and then call the MCP tools in the order the skill prescribes.")

### Your turn
1. Add a second skill, `travel-booking`, with only frontmatter and three steps. Re-run section 4: how much did level 1 grow, and did level 2 for `expense-review` change?
2. Break the skill: rename it to `Expense_Review` in the frontmatter and re-run section 4. Which rule of the Agent Skills spec fails?
3. Move the 25 % escalation rule out of `SKILL.md` and into the MCP tool (return a `verdict` field instead). Which design is better if three different agents use this server but only one company has that rule? Argue both sides in two sentences each.
4. In one sentence each, explain function calling, MCP and Skills to a non-technical manager using the employee analogy, then time yourself saying all three in under 30 seconds.